# Task 3 — reproducible analysis

This notebook separates the synthetic benchmark scaffold from the real classroom analysis. Only ProgFeed results are empirical student outcomes. The dataset does not measure critical thinking.

## 1. Environment and reproducibility

Fixed seed: 42. Install requirements.txt and run src/fetch_data.py before running all cells.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import process_data as proc
import visualize as benchmark_viz
from analyze_progfeed import submission_outcomes, summarize
from visualize_progfeed import make_figures

RAW_BENCH = ROOT / 'data/raw/test_run.jsonl'
RAW_STUDY = ROOT / 'data/raw/progfeed/all_submissions_consolidated.csv'
print('Project:', ROOT)
print('Seed:', SEED)
print('Raw study data available:', RAW_STUDY.exists())

Project: C:\Users\game0\Downloads\methods\Methods_Adi_Alan
Seed: 42
Raw study data available: True


## 2. Benchmark example: ingestion and warm-up filtering

This benchmark times Python-loop and pandas aggregation over actual ProgFeed rows. Warm-up timings are discarded before summary; figures 1–2 show local ETL measurements, not AI API latency.

In [2]:
df_raw = proc.load_jsonl(RAW_BENCH)
print('Raw measured runs:', len(df_raw))
print('Phases before filtering:')
print(df_raw['phase'].value_counts().to_string())
df_bench = proc.filter_warmup(df_raw, phase_col='phase')
print('Rows after warm-up filtering:', len(df_bench))

2026-11-19 23:41:23,688 | INFO | Loaded 420 measured runs from C:\Users\game0\Downloads\methods\Methods_Adi_Alan\data\raw\test_run.jsonl


2026-11-19 23:41:23,691 | INFO | Warm-up filter: 420 total -> 360 measured (60 warm-up removed)


Raw measured runs: 420
Phases before filtering:
phase
benchmark    360
warmup        60
Rows after warm-up filtering: 360


In [3]:
benchmark_summary = proc.compute_summary(df_bench, seed=SEED)
benchmark_summary.to_csv(ROOT / 'data/processed/metrics_summary.csv', index=False)
benchmark_viz.fig1_latency_vs_load(benchmark_summary)
benchmark_viz.fig2_distribution_boxplot(df_bench)
print('Measured pipeline exports: fig1_latency_vs_load and fig2_distribution_boxplot (PDF + PNG)')
display(benchmark_summary[['variant', 'load_level', 'n_total', 'mean', 'median', 'p95', 'p99', 'error_rate']].head(6))

Measured pipeline exports: fig1_latency_vs_load and fig2_distribution_boxplot (PDF + PNG)


,variant,load_level,n_total,mean,median,p95,p99,error_rate
0,Baseline (Python row loop),100,30,0.428153,0.34455,0.776540,0.939594,0.0
1,Proposed (pandas groupby),100,30,3.534363,3.40825,4.622215,5.036944,0.0
2,Baseline (Python row loop),250,30,0.775010,0.74940,1.093430,1.231659,0.0
3,Proposed (pandas groupby),250,30,4.147577,3.75180,5.706735,6.171962,0.0
4,Baseline (Python row loop),500,30,1.120170,0.98045,1.721075,1.941208,0.0
5,Proposed (pandas groupby),500,30,3.580740,3.35160,4.667390,4.759456,0.0


## 3. ProgFeed data ingestion and filtering

The source contains one row per graded test/function per submission. Comparisons are restricted to labs 05, 06, 07, and 09, where feedback assignment was randomized. Test rows are aggregated into submissions and then into student-lab trajectories. Feedback assignment and actual feedback delivery are tracked separately.

In [4]:
source = pd.read_csv(RAW_STUDY, keep_default_na=False)
units = submission_outcomes(RAW_STUDY)
print('Source rows:', len(source))
print('Consenting students:', source.loc[source['consented'].astype(str).str.lower() == 'true', 'student_id'].nunique())
print('Randomized-lab student units:', len(units))
print('Arms:', units['feedback_type'].value_counts().to_dict())
print('First submission failed:', int(units['first_attempt_failed'].sum()))

Source rows: 17385
Consenting students: 215
Randomized-lab student units: 803
Arms: {'no_feedback': 271, 'tc': 268, 'nl': 264}
First submission failed: 280


## 4. Outcome summaries and uncertainty

For each initially failing student-lab unit, score gain is the change from its first to final observed submission. Retry and eventual full-credit rates are also reported. The 95% interval resamples student IDs and keeps each sampled student's lab records together; the seed is fixed.

In [5]:
student_summary = summarize(units, seed=SEED)
student_summary.to_csv(ROOT / 'data/processed/student_outcomes_summary.csv', index=False)
display(student_summary.round(3))

,feedback_type,student_lab_units,first_attempt_failed,retry_rate_after_first_failure,feedback_delivered_units,eventual_full_pass_rate_after_first_failure,mean_score_gain_pp_after_first_failure,score_gain_cluster_ci95_low,score_gain_cluster_ci95_high
0,no_feedback,271,93,0.978,0,0.914,64.552,58.478,70.538
1,tc,268,91,0.956,71,0.901,63.187,56.159,70.366
2,nl,264,96,0.969,84,0.927,68.507,61.546,74.830


## 5. Scientific visualization and export

Figures 3–4 use the empirical ProgFeed outcomes. They show score gain with student-cluster bootstrap intervals and the distribution of final submission scores. Exports are PDF and PNG at 300 DPI.

In [6]:
make_figures(RAW_STUDY)
fig_dir = ROOT / 'reports/figures'
print('Empirical exports:')
print((fig_dir / 'fig3_feedback_score_gain.pdf').exists(), (fig_dir / 'fig3_feedback_score_gain.png').exists())
print((fig_dir / 'fig4_final_score_distribution.pdf').exists(), (fig_dir / 'fig4_final_score_distribution.png').exists())

Empirical exports:
True True
True True


## 6. Interpretation and limits

These are descriptive assignment-arm summaries, not evidence about critical thinking. The dataset contains no validated critical-thinking measure. Hints are delivered only after failed attempts, outcomes repeat within students, and labs outside the four randomized interventions are not part of arm comparisons. Do not read the synthetic benchmark values in section 2 as classroom or production measurements.